# Homework04 - Task 5 Node Classification

This notebook is based on:
- S06-15 GML Node Classification
- MSD repository usage pattern in S0 Classes/msd-main

Goal: run node classification on your recreated graph dataset and analyze correct and incorrect predictions.

## Cell 2 - Imports

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from topologicpy.PyG import PyG
from topologicpy.Helper import Helper

In [ ]:
print("TopologicPy version:", Helper.Version())

## Cell 5 - Set paths

Create your final dataset folder with graphs.csv, nodes.csv, edges.csv under Homework04/Notebooks.

In [ ]:
BASE = Path(r"C:\Users\etmaglari\IAAC\etmaglari_gML")
DATASET_PATH = BASE / "Homework04" / "Notebooks" / "dataset_node_classification_custom"
REFERENCE_PATH = BASE / "example_dataset" / "dataset_node_classification"
OUT_DIR = BASE / "Homework04" / "Notebooks" / "classification_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Custom dataset path:", DATASET_PATH)
print("Reference dataset path:", REFERENCE_PATH)

## Cell 7 - Check required files

In [ ]:
def check_dataset(path):
    required = [path / "graphs.csv", path / "nodes.csv", path / "edges.csv"]
    ok = True
    for f in required:
        exists = f.exists()
        print(f"{f.name}: {'FOUND' if exists else 'MISSING'} -> {f}")
        ok = ok and exists
    return ok

HAS_CUSTOM = check_dataset(DATASET_PATH)
if not HAS_CUSTOM:
    print("Custom dataset missing. Temporarily using reference dataset for pipeline testing.")
    DATASET_PATH = REFERENCE_PATH
    _ = check_dataset(DATASET_PATH)

## Cell 9 - Inspect schema

In [ ]:
graphs_df = pd.read_csv(DATASET_PATH / "graphs.csv")
nodes_df = pd.read_csv(DATASET_PATH / "nodes.csv")
edges_df = pd.read_csv(DATASET_PATH / "edges.csv")

print("graphs.csv shape:", graphs_df.shape)
print("nodes.csv shape:", nodes_df.shape)
print("edges.csv shape:", edges_df.shape)
print()
print("graphs columns:", list(graphs_df.columns))
print("nodes columns:", list(nodes_df.columns))
print("edges columns:", list(edges_df.columns))

In [ ]:
feature_cols = [c for c in nodes_df.columns if c.startswith("feat_")]
summary = {
    "num_graphs": int(graphs_df["graph_id"].nunique()) if "graph_id" in graphs_df.columns else None,
    "num_nodes": int(len(nodes_df)),
    "num_edges": int(len(edges_df)),
    "num_features": int(len(feature_cols)),
    "has_train_mask": bool("train_mask" in nodes_df.columns),
    "has_val_mask": bool("val_mask" in nodes_df.columns),
    "has_test_mask": bool("test_mask" in nodes_df.columns)
}
print(json.dumps(summary, indent=2))

## Cell 12 - Build PyG object and configure hyperparameters

This follows S06-15 baseline settings and can be compared with seminar outputs.

In [ ]:
pyg = PyG.ByCSVPath(
    path=str(DATASET_PATH),
    level="node",
    task="classification",
    graphLabelType="categorical",
    nodeLabelType="categorical",
    edgeLabelType="categorical"
)

pyg.SetHyperparameters(
    cv="holdout",
    split=(0.7, 0.15, 0.15),
    random_state=42,
    shuffle=True,
    epochs=50,
    batch_size=10,
    lr=1e-2,
    weight_decay=0.01,
    optimizer="adamw",
    gradient_clip_norm=1.0,
    early_stopping=True,
    early_stopping_patience=12,
    use_gpu=True,
    conv="sage",
    hidden_dims=(64, 64, 64),
    activation="relu",
    dropout=0.0,
    batch_norm=True,
    residual=False,
    pooling="mean"
)

print("PyG object ready.")

## Cell 14 - Train, Validate, Test

In [ ]:
history = pyg.Train()
val_metrics = pyg.Validate()
test_metrics = pyg.Test()

print("Validation metrics:")
print(pd.Series(val_metrics))
print()
print("Test metrics:")
print(pd.Series(test_metrics))

In [ ]:
fig_hist = pyg.PlotHistory()
fig_hist.update_layout(width=900, height=700)
fig_hist.show()

## Cell 17 - Export node predictions and analyze mistakes

In [ ]:
pred_report = pyg.Predict(split="all", return_probs=True, attach_to_data=True)

pred_csv = OUT_DIR / "node_predictions_homework04.csv"
rows = []
for gi, data in enumerate(pyg.data_list):
    graph_id = int(data.graph_id.item()) if hasattr(data, "graph_id") else gi
    y_true = np.asarray(pred_report["y_true"][gi]).squeeze()
    y_pred = np.asarray(pred_report["pred"][gi]).squeeze()

    if y_true.ndim > 1:
        y_true = np.argmax(y_true, axis=1)
    if y_pred.ndim > 1:
        y_pred = np.argmax(y_pred, axis=1)

    train_mask = data.train_mask.detach().cpu().numpy() if hasattr(data, "train_mask") else [False] * len(y_true)
    val_mask = data.val_mask.detach().cpu().numpy() if hasattr(data, "val_mask") else [False] * len(y_true)
    test_mask = data.test_mask.detach().cpu().numpy() if hasattr(data, "test_mask") else [False] * len(y_true)

    for ni in range(len(y_true)):
        rows.append({
            "graph_id": graph_id,
            "node_id": int(ni),
            "y_true": int(y_true[ni]),
            "y_pred": int(y_pred[ni]),
            "correct": bool(int(y_true[ni]) == int(y_pred[ni])),
            "train_mask": bool(train_mask[ni]),
            "val_mask": bool(val_mask[ni]),
            "test_mask": bool(test_mask[ni])
        })

pred_df = pd.DataFrame(rows)
pred_df.to_csv(pred_csv, index=False)

print("Saved predictions to:", pred_csv)
print("Overall accuracy:", round(pred_df["correct"].mean(), 4))
display(pred_df.head(20))

In [ ]:
print("Error analysis by split:")
for split_col in ["train_mask", "val_mask", "test_mask"]:
    sub = pred_df[pred_df[split_col]]
    if len(sub) == 0:
        continue
    acc = sub["correct"].mean()
    print(f"{split_col}: n={len(sub)}, accuracy={acc:.4f}")

print()
print("Top confusion pairs on test split:")
test_err = pred_df[(pred_df["test_mask"]) & (~pred_df["correct"])].copy()
if len(test_err) == 0:
    print("No test errors found.")
else:
    conf = test_err.groupby(["y_true", "y_pred"]).size().reset_index(name="count")
    conf = conf.sort_values("count", ascending=False)
    display(conf.head(10))

## Cell 20 - Presentation interpretation checklist

For your final slides, summarize:
1. Predicted node classes and global accuracy
2. Correct predictions that are spatially coherent
3. Incorrect predictions and likely causes (feature overlap, sparse context, mislabeled training samples)
4. Whether mistakes happen in boundary spaces such as corridor, entrance, or service rooms
5. Two concrete improvements for the next iteration

## Presentation Script Block

Use this section to write your final narrative for Task 5 in slide-ready form.

### Setup
- Dataset path used.
- Model setup used (baseline training or pretrained inference).
- Number of graphs, nodes, and classes.

### Results
- Validation and test metrics.
- Overall node-level accuracy.
- Most frequent confusion pairs on test nodes.

### Spatial Meaning of Predictions
- 2 examples of correct predictions that make spatial sense.
- 2 examples of incorrect predictions and why they may fail.
- Explain whether mistakes happen at boundaries, transition spaces, or underrepresented classes.

### Improvement Plan
- Data-centric: class balance, additional labels, cleaner masks.
- Model-centric: architecture tuning or regularization changes.
- Graph-centric: improved features from geometry, adjacency type, or floor-level context.